# ALOHA Demo (Colab) — No-Teleop VLA + RL Pipeline

This notebook mirrors the quadruped Colab workflow, but for the **ALOHA move-blue-ball demo** in Solace Robotics.

It runs the current pipeline stages:
1. Teacher data generation (scaffold)
2. VLA student bootstrap (scaffold)
3. RL refinement (scaffold)
4. Evaluation + report artifacts

> Note: this is a production scaffold ready for simulator/trainer backend plug-in.


In [ ]:
#@title 1) Clone / update Solace Robotics repo
import os, subprocess, textwrap

REPO_URL = 'https://github.com/Solace-Stephane/solace-robotics.git'
REPO_DIR = '/content/solace-robotics'

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

print("Repo ready:", REPO_DIR)


In [ ]:
#@title 2) Install dependencies
%cd /content/solace-robotics
!python -m pip -q install -r pipelines/requirements.txt pandas matplotlib
print("Dependencies installed.")


In [ ]:
#@title 3) Run full pipeline (base IL -> RL refine -> eval)
import subprocess

commands = [
    "python pipelines/scripts/generate_teacher_data.py --config pipelines/configs/base_il.yaml",
    "python pipelines/scripts/train_vla_student.py --config pipelines/configs/base_il.yaml",
    "python pipelines/scripts/rl_refine_student.py --config pipelines/configs/rl_refine.yaml",
    "python pipelines/scripts/run_eval.py --config pipelines/configs/eval.yaml",
]

for cmd in commands:
    print("\n>>>", cmd)
    subprocess.run(cmd, shell=True, check=True)

print("\nPipeline run complete.")


In [ ]:
#@title 4) Inspect generated artifacts
import json
from pathlib import Path

artifact_root = Path('/content/solace-robotics/pipelines/artifacts/move_blue_ball')
eval_scores = artifact_root / 'eval' / 'scores.json'
eval_summary = artifact_root / 'eval' / 'summary.md'

print("Artifact root:", artifact_root)
for p in sorted(artifact_root.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(artifact_root))

print("\n=== scores.json ===")
print(eval_scores.read_text())

print("\n=== summary.md ===")
print(eval_summary.read_text())


In [ ]:
#@title 5) Quick metrics view
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

scores_path = Path('/content/solace-robotics/pipelines/artifacts/move_blue_ball/eval/scores.json')
scores = json.loads(scores_path.read_text())

rows = [
    ["standard_success_rate", scores["standard_success_rate"], scores["acceptance"]["standard_success_rate_min"]],
    ["hard_success_rate", scores["hard_success_rate"], scores["acceptance"]["hard_success_rate_min"]],
]

df = pd.DataFrame(rows, columns=["metric", "value", "target"])
display(df)

ax = df.plot(x="metric", y=["value", "target"], kind="bar", figsize=(8,4), title="Move Blue Ball Eval")
ax.set_ylim(0, 1.0)
ax.set_ylabel("Rate")
plt.show()

print("Acceptance pass:", scores["acceptance"]["pass"])


## Keyboard-style control panel (quadruped-like UX)

This panel is included to mirror the keyboard feel from the quadruped notebook.
For now, it adjusts task knobs for demo and can trigger eval reruns.
When the simulator backend is plugged in, this same panel can map to live command inputs.


In [ ]:
#@title 6) Keyboard panel (WASD + R)
from google.colab import output
from IPython.display import HTML, Javascript, display
import subprocess, textwrap

state = {"target_x": 0.44, "target_y": 0.00, "spawn_noise": 0.10}

def clip(v, lo, hi):
    return max(lo, min(hi, v))

def status_html():
    return f"""
    <div style="font-family: monospace; padding:10px; border:1px solid #bbb; border-radius:8px; margin-top:8px;">
      target_x: {state["target_x"]:+.2f} &nbsp; target_y: {state["target_y"]:+.2f} &nbsp; spawn_noise: {state["spawn_noise"]:.2f}
    </div>
    """

def rerun_eval():
    cmd = "python pipelines/scripts/run_eval.py --config pipelines/configs/eval.yaml"
    print("Running:", cmd)
    subprocess.run(cmd, shell=True, check=True)
    print("Eval rerun complete.")

def on_key(key):
    k = (key or "").lower()
    if k == "w":
        state["target_y"] = clip(state["target_y"] + 0.01, -0.24, 0.24)
    elif k == "s":
        state["target_y"] = clip(state["target_y"] - 0.01, -0.24, 0.24)
    elif k == "a":
        state["target_x"] = clip(state["target_x"] - 0.01, 0.28, 0.60)
    elif k == "d":
        state["target_x"] = clip(state["target_x"] + 0.01, 0.28, 0.60)
    elif k == "q":
        state["spawn_noise"] = clip(state["spawn_noise"] - 0.01, 0.00, 0.30)
    elif k == "e":
        state["spawn_noise"] = clip(state["spawn_noise"] + 0.01, 0.00, 0.30)
    elif k == "r":
        rerun_eval()
    elif k == " ":
        state.update({"target_x": 0.44, "target_y": 0.00, "spawn_noise": 0.10})
    display(HTML(status_html()))

output.register_callback('aloha.keypress', on_key)

display(HTML("""
<div id="kbx" tabindex="0" style="padding:12px; border:2px dashed #777; border-radius:8px; outline:none;">
  <b>Click here, then use keyboard:</b> W/S/A/D move target, Q/E noise, Space reset, R rerun eval
</div>
"""))

display(Javascript("""
const el = document.getElementById("kbx");
el.focus();
el.addEventListener("keydown", async (e) => {
  e.preventDefault();
  await google.colab.kernel.invokeFunction("aloha.keypress", [e.key], {});
});
"""))

display(HTML(status_html()))


## Next step (when backend is connected)

- Replace scaffold script internals with simulator + trainer runtime hooks.
- Keep this notebook unchanged as the demo/operator UX layer.
